# EDA 
This notebook explores the data evailable transit data for further development


## Packages

In [1]:
import os 
from google.transit import gtfs_realtime_pb2
import requests
from requests.exceptions import Timeout, ConnectionError, HTTPError
from time import sleep
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from typing import Optional, List
import mysql.connector
import geopandas as gpd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Constants

In [2]:
realtime_vehicle_updates_url = "https://bct.tmix.se/gtfs-realtime/vehicleupdates.pb?operatorIds=47"

root_dir = os.path.dirname(os.path.abspath("."))  # Fixed typo from dirname to dir
data_dir = os.path.join(root_dir, "data")
src_dir = os.path.join(root_dir, "src")
os.makedirs(data_dir, exist_ok=True)

df_path = os.path.join(data_dir, "vehicle_updates.csv")
df_bus_stop_path = os.path.join(data_dir, "Kelowna_Regional_Transit_System_stops.csv")

## Fetch data 

In [3]:

table_names = ["TRANSIT_VEHICLE_TABLE", "TRANSIT_TRIP_TABLE"]
conn = mysql.connector.connect(
    host="10.0.0.245",
    user="transit",
    password="transit123",
    database="transitdb",
    port=3307
)
for table in table_names:
    cursor = conn.cursor()
    cursor.execute(f"ANALYZE TABLE transitdb.{table}")
    result = cursor.fetchall()
    print(result)

    
    df_size = pd.read_sql(f"""
        SELECT
            table_name,
            ROUND((data_length + index_length) / 1024 / 1024, 2) AS size_mb
        FROM information_schema.tables
        WHERE table_schema = 'transitdb'
          AND table_name = '{table}'
    """, conn)

    print(df_size)
    cursor.close()

conn.close()


[('transitdb.TRANSIT_VEHICLE_TABLE', 'analyze', 'status', 'OK')]
              TABLE_NAME  size_mb
0  TRANSIT_VEHICLE_TABLE   2040.7


/tmp/ipykernel_18605/1053477817.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_size = pd.read_sql(f"""


[('transitdb.TRANSIT_TRIP_TABLE', 'analyze', 'status', 'OK')]
           TABLE_NAME   size_mb
0  TRANSIT_TRIP_TABLE  14261.98


/tmp/ipykernel_18605/1053477817.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_size = pd.read_sql(f"""
